In [1]:
import numpy as np
import pandas as pd
import ast
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [2]:
df = pd.read_csv('final_data.csv')
df['text'] = df['text'].apply(ast.literal_eval)
df['next_word'] = df['next_word'].apply(ast.literal_eval)

In [3]:
x = df['text'].values
y = df['next_word'].values

x = [torch.tensor(i, dtype=torch.long) for i in x]
y = [torch.tensor(i, dtype=torch.long) for i in y]

In [4]:
class NextWordDataset(TensorDataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

In [5]:
dataset = NextWordDataset(x, y)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [6]:
class LSTMModel(torch.nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(LSTMModel, self).__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embedding_dim)
        self.lstm = torch.nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        # output size = vocab size
        self.fc = torch.nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, _ = self.lstm(embedded)
        lstm_out = lstm_out[:, -1, :]
        output = self.fc(lstm_out)
        return output


In [7]:
vocab_size = 3040
embedding_dim = 128
hidden_dim = 256

model = LSTMModel(
    vocab_size,
    embedding_dim,
    hidden_dim
)

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [8]:
for epoch in range(30):
    model.train()
    total_loss = 0
    for batch_x, batch_y in dataloader:
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y.squeeze())

        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'Epoch {epoch+1}, Loss: {total_loss/len(dataloader)}')

Epoch 1, Loss: 5.943494597750325
Epoch 2, Loss: 4.3043765537841345
Epoch 3, Loss: 3.473057125204353
Epoch 4, Loss: 2.9241425672205548
Epoch 5, Loss: 2.5805200359193226
Epoch 6, Loss: 2.3529523611068726
Epoch 7, Loss: 2.1527988458833387
Epoch 8, Loss: 2.0292621159585575
Epoch 9, Loss: 1.922496197845346
Epoch 10, Loss: 1.8837194143924663
Epoch 11, Loss: 1.8332678859112084
Epoch 12, Loss: 1.7777325586285642
Epoch 13, Loss: 1.7616360060149623
Epoch 14, Loss: 1.737104408843543
Epoch 15, Loss: 1.7152184191890942
Epoch 16, Loss: 1.6702617015088759
Epoch 17, Loss: 1.6378286097158667
Epoch 18, Loss: 1.6231278594463103
Epoch 19, Loss: 1.6303372830152512
Epoch 20, Loss: 1.6036918680033376
Epoch 21, Loss: 1.5985500109932755
Epoch 22, Loss: 1.559116298633237
Epoch 23, Loss: 1.539450055208578
Epoch 24, Loss: 1.5633039572226104
Epoch 25, Loss: 1.5590932668857678
Epoch 26, Loss: 1.5877066329922727
Epoch 27, Loss: 1.52277876004096
Epoch 28, Loss: 1.4766308547908902
Epoch 29, Loss: 1.4906629555488145
Ep

In [9]:
import json
import torch

with open('word_to_id.json', 'r') as f:
    id = json.load(f)

with open('id_to_word.json', 'r') as f:
    word_id = json.load(f)

def predict_next_word(text):

    tokens = text.split()

    seq = [id[word] for word in tokens if word in id]
    seq = torch.tensor(seq).unsqueeze(0)

    pad_len = 19 - len(seq)

    seq = nn.functional.pad(
        seq,
        (pad_len, 0),
        value=0
    )

    with torch.no_grad():

        pred = model(seq)

        pred_idx = torch.argmax(pred, dim=1).item()

    return word_id.get(str(pred_idx), "<unk>")
predict_next_word("i hope this email")

'finds'

In [10]:
text = "i appreciate your valuable"
for i in range(1,5):
    result = predict_next_word(text)
    text = text+" "+result

print(text)

i appreciate your valuable insights through attending conferences


In [11]:
torch.save(model.state_dict(), "lstm_next_word.pth")